# Sorghum-100 Cultivar Identification (FGVC 9)

Code submitted to the Kaggle competition [Sorghum-100 Cultivar Identification - FGVC 9](https://www.kaggle.com/competitions/sorghum-id-fgvc-9):
classify aerial RGB images of sorghum plants into one of 100 cultivars.

**Result:** the submission scored **0.743 on the public leaderboard and 0.73 on the private leaderboard**.
The competition metric is top-1 classification accuracy on the hidden test set.

**Approach:** an ImageNet-pretrained EfficientNetB2, fully fine-tuned at 600x600 px with standard augmentation plus CutMix
and a cyclical learning rate.

**Notes**
- Runs as a Kaggle notebook with a GPU. It reads the images from the `small-jpegs-fgvc` Kaggle dataset
  (the competition images re-encoded as smaller JPEGs), mounted under `DATA_DIR` (see the repository `README.md`).
- No validation split is used: the whole training set is used for fitting, so the accuracy reported during training
  is *training* accuracy (on CutMix-augmented batches), not a held-out estimate.

## 1. Importing and Installing Dependencies

In [ ]:
!pip install -q cutmix-keras

In [ ]:
import os

import pandas as pd
import tensorflow as tf
import tensorflow_addons as tfa
from cutmix_keras import CutMixImageDataGenerator
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import CSVLogger

## 2. Setting the Hyperparameters

In [ ]:
# PATHS
DATA_DIR = "../input/small-jpegs-fgvc"  # Kaggle dataset with train/, test/ and train_cultivar_mapping.csv

# HYPERPARAMETERS
IMAGE_SIZE = (600, 600, 3)
BATCH_SIZE = 15
EPOCHS = 10

# EXPONENTIAL DECAY (alternative schedule, disabled)
USE_DECAY = False
LEARNING_RATE = 0.0001
DECAY_RATE = 0.9

# CYCLICAL LEARNING RATE
USE_CYCLICAL = True
INITIAL_LR = 8e-5
MAX_LR = 4e-4

# EARLY STOPPING: stop once training accuracy exceeds this threshold
ES_ACC = 0.9

# FINE-TUNING: freeze the first FINE_TUNING_LAYERS layers of the backbone when enabled
FINE_TUNE = False
FINE_TUNING_LAYERS = 0

## 3. Importing the Dataset

Images are read from disk with Keras generators and augmented on the fly (shear, zoom, horizontal flip, rotation,
brightness and shifts). On top of that, **CutMix** pastes a random square patch from an image of a second, independently
shuffled generator into each image, and mixes the one-hot labels in proportion to the patch area. This is a strong
regularizer for a fine-grained task where all classes look very similar.

In [ ]:
dtf = pd.read_csv(f"{DATA_DIR}/train_cultivar_mapping.csv")

train_datagen = ImageDataGenerator(shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   fill_mode='reflect',
                                   rotation_range=25,
                                   brightness_range=(0.8, 1.2),
                                   width_shift_range=0.1,
                                   height_shift_range=0.1)

# Two independently shuffled generators over the same images: CutMix combines one batch from each
training_set1 = train_datagen.flow_from_dataframe(dataframe=dtf,
                                                  directory=f"{DATA_DIR}/train",
                                                  x_col="image",
                                                  y_col="cultivar",
                                                  target_size=IMAGE_SIZE[0:2],
                                                  batch_size=BATCH_SIZE)
training_set2 = train_datagen.flow_from_dataframe(dataframe=dtf,
                                                  directory=f"{DATA_DIR}/train",
                                                  x_col="image",
                                                  y_col="cultivar",
                                                  target_size=IMAGE_SIZE[0:2],
                                                  batch_size=BATCH_SIZE)

training_set = CutMixImageDataGenerator(
    generator1=training_set1,
    generator2=training_set2,
    img_size=IMAGE_SIZE[0],
    batch_size=BATCH_SIZE,
)

total_steps = len(training_set1)

## 4. Importing the Base Model - EfficientNetB2

In [ ]:
base_model = tf.keras.applications.efficientnet.EfficientNetB2(include_top=False, weights='imagenet', input_shape=IMAGE_SIZE)
base_model.trainable = True
print("Number of layers in the base model: ", len(base_model.layers))

In [ ]:
if FINE_TUNE:
    print("FINE-TUNING")
    fine_tune_at = FINE_TUNING_LAYERS

    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

## 5. Building the Model

The backbone is followed by global average pooling, dropout and a 100-way softmax head. The backbone is called with
`training=False`, so its BatchNorm layers stay in inference mode while all weights are updated.

In [ ]:
inputs = tf.keras.Input(shape=IMAGE_SIZE)
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(100, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

Setting the learning-rate schedule: a triangular cyclical learning rate whose amplitude halves every cycle (exponential decay is kept as an alternative).

In [ ]:
if USE_CYCLICAL:
    LR = tfa.optimizers.CyclicalLearningRate(initial_learning_rate=INITIAL_LR,
                                             maximal_learning_rate=MAX_LR,
                                             scale_fn=lambda x: 1 / (2.**(x - 1)),
                                             step_size=2 * total_steps)

elif USE_DECAY:
    LR = tf.keras.optimizers.schedules.ExponentialDecay(
        LEARNING_RATE,
        decay_steps=total_steps,
        decay_rate=DECAY_RATE,
        staircase=True)

## 6. Setting the Callbacks

In [ ]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    'my_model',
    monitor='loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=True)

model_checkpoint_callback.set_model(model)


class StopAtAccuracy(tf.keras.callbacks.Callback):
    """Stop training once the training accuracy exceeds ES_ACC."""

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if logs.get('accuracy', 0) > ES_ACC:
            self.model.stop_training = True


early_stopping = tf.keras.callbacks.EarlyStopping(monitor='accuracy', patience=3)

csv_log = CSVLogger("results.csv")

## 7. Compiling and Training

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
              metrics=['accuracy'])
history = model.fit(training_set,
                    epochs=EPOCHS,
                    callbacks=[model_checkpoint_callback, StopAtAccuracy(), early_stopping, csv_log],
                    steps_per_epoch=total_steps,
                    verbose=2)

## 8. Predicting and Submitting

The test images are read in a fixed order (`shuffle=False`), predicted, mapped back from class indices to cultivar names,
and written to `submission.csv`. File names are converted back to the `.png` names expected by the competition.

In [ ]:
test_images = tf.keras.utils.image_dataset_from_directory(f"{DATA_DIR}/test",
                                                          labels=None,
                                                          label_mode=None,
                                                          batch_size=BATCH_SIZE,
                                                          image_size=IMAGE_SIZE[0:2],
                                                          shuffle=False)

In [ ]:
predictions = model.predict(test_images)
predictions = tf.argmax(predictions, axis=1).numpy()

In [ ]:
paths = [os.path.basename(path).replace('.jpeg', '.png') for path in test_images.file_paths]
indices = {value: key for key, value in training_set.class_indices.items()}

In [ ]:
sub = pd.DataFrame({'filename': paths, 'cultivar': predictions})
sub['cultivar'] = sub.cultivar.map(indices)
sub.to_csv("submission.csv", index=False)